# 🚁 Notebook de Simulation : Drone Tello EDU Explorateur

Ce notebook permet d'exécuter le système d'exploration autonome basé sur le code fourni. Il utilise le mode simulation intégré pour générer des flux vidéo synthétiques et simuler la physique du drone.

### 1. Installation des dépendances
Installez les bibliothèques nécessaires listées dans `requirements.txt`.

In [ ]:
# Installation des dépendances
!pip install djitellopy numpy opencv-python matplotlib

### 2. Importation et Configuration
Assurons-nous que Python peut trouver les modules. Si vous avez téléchargé les fichiers dans un dossier spécifique, ajustez le chemin ci-dessous.

In [ ]:
import sys
import os
import time
import cv2
import numpy as np
import matplotlib.pyplot as plt
from threading import Thread

# Ajoutez le chemin vers vos fichiers si nécessaire
# sys.path.append('/content/votre_dossier') 

# Importation des modules du projet
from tello_controller import TelloController
from exploration import ExplorationMission, MissionConfig
from vision import VideoStream, ObstacleDetector
from visual_slam import VisualSLAM

# Fonction utilitaire pour afficher les images dans le notebook
def show_image(img, title="Image"):
    plt.figure(figsize=(10, 6))
    if len(img.shape) == 3:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    else:
        plt.imshow(img, cmap='gray')
    plt.title(title)
    plt.axis('off')
    plt.show()

print("Modules importés avec succès.")

### 3. Test du Contrôleur (Mode Simulation)
Testons d'abord les mouvements basiques du drone via `tello_controller.py`. En mode simulation, cela mettra à jour la position interne sans connecter de vrai drone.

In [ ]:
print("--- Test du Contrôleur ---")
controller = TelloController(simulation_mode=True)
controller.connect()
controller.takeoff()

# Effectuer quelques mouvements
controller.move_up(50)
controller.move_forward(100)
controller.rotate_clockwise(90)

# Afficher la télémétrie
telemetry = controller.get_telemetry()
print(f"Position actuelle : {telemetry['position']}")
print(f"Batterie : {telemetry['battery']}%")

controller.land()

### 4. Test de la Vision et Détection d'Obstacles
Le module `vision.py` génère des images synthétiques en mode simulation. Nous allons récupérer une frame et tester le détecteur d'obstacles.

In [ ]:
print("--- Test Vision & Obstacles ---")

# Initialisation
video = VideoStream(simulation_mode=True)
detector = ObstacleDetector()

video.start()
time.sleep(1) # Laisser le temps au thread de démarrer

# Récupérer une frame
frame = video.get_frame()

if frame is not None:
    # Détection
    obstacles = detector.detect(frame)
    
    # Dessiner les résultats
    result_img = detector.draw_detections(frame, obstacles)
    
    # Afficher
    print(f"Obstacles détectés : {len(obstacles)}")
    show_image(result_img, "Vue Drone (Simulée) + Détections")
else:
    print("Erreur: Pas de frame reçue")

video.stop()

### 5. Exécution d'une Mission Complète (SLAM + Exploration)
Nous allons lancer une mission autonome complète utilisant `ExplorationMission` et `VisualSLAM`.

Nous allons :
1. Configurer une zone de 300x300 cm.
2. Ajouter des obstacles virtuels.
3. Lancer la mission pendant 10 secondes.
4. Visualiser la carte générée.

In [ ]:
# Configuration de la mission
config = MissionConfig(
    area_width=300,
    area_height=300,
    exploration_altitude=100,
    step_size=50,
    pattern="snake",     # Pattern en serpent
    enable_mapping=True,
    enable_avoidance=True
)

# Création de la mission
mission = ExplorationMission(config, simulation_mode=True)

# Ajout d'obstacles simulés pour tester l'évitement
# Obstacle fixe
mission.add_simulated_obstacle(100, 50, 100, is_mobile=False)
# Obstacle mobile avec vélocité
mission.add_simulated_obstacle(0, 100, 100, is_mobile=True, velocity=(10, 5, 0))

print("Préparation de la mission...")
if mission.prepare_mission():
    print("Démarrage de l'exploration...")
    mission.start_exploration()
    
    # Simulation du temps qui passe (laisser tourner 10 secondes)
    # En temps réel, le drone bougerait et mettrait à jour sa carte
    try:
        for i in range(10):
            pos = mission.controller.position
            prog = mission.planner.get_progress()
            print(f"Sec {i+1}/10 - Pos: ({pos.x:.0f}, {pos.y:.0f}) - Prog: {prog:.1f}%")
            time.sleep(1)
    except KeyboardInterrupt:
        print("Interruption utilisateur")
    
    print("Arrêt de la mission...")
    mission.stop_exploration()
    
    # Affichage de la carte ASCII
    print("\n--- Carte d'Altitude (ASCII) ---")
    mission.display_map()
    
    # Récupération du rapport
    report = mission.get_mission_report()
    print(f"\nStatistiques finales :")
    print(f"Waypoints atteints : {report['waypoints']['completed']}")
    print(f"Obstacles vus : {report['obstacles']['total_obstacles']}")

else:
    print("Échec de la préparation de la mission")

### 6. Visualisation Graphique de la Carte SLAM
Le code `visual_slam.py` génère une grille d'occupation (`occupancy_grid`). Visualisons-la avec Matplotlib pour un rendu plus propre que l'ASCII.

In [ ]:
# Initialisation du SLAM en mode simulation pour visualiser la logique
slam = VisualSLAM(simulation_mode=True)
slam.start()

# Laisser le SLAM accumuler quelques données (simulées)
print("Accumulation de données SLAM...")
time.sleep(2)

# Récupérer la grille
grid = slam.occupancy_grid
slam.stop()

# Visualisation
plt.figure(figsize=(10, 8))
# -1 = inconnu, 0 = libre, 100 = occupé
plt.imshow(grid, cmap='viridis', interpolation='nearest')
plt.colorbar(label='Probabilité d\'occupation')
plt.title("Carte SLAM (Grille d'occupation)")
plt.xlabel("X (cellules)")
plt.ylabel("Y (cellules)")
plt.show()

### 7. Export des Données
Le système permet d'exporter les résultats en JSON et CSV.

In [ ]:
output_dir = "mission_results"
mission.export_results(output_dir)

print(f"Fichiers exportés avec le préfixe '{output_dir}' :")
!ls {output_dir}*